# Assignment 3: Model Evaluation
### COMM 4259 <br> <i>Foundations of Machine Learning and AI with Python</i>

© Jingjing Li- Ask for permission for using/quoting: jingjing.li@virginia.edu


# Description

This assignment has 4 sections and is to be completed individually. Collaborations with peers or seeking assistance from unauthorized sources will be considered a breach of the Honor Code.

| Section | What | Weight | Generative AI |
|---|---|---|---|
| 1 | Compare validation methods | 25% | Not allowed |
| 2 | Cost-benefit analysis | 20% | Not allowed |
| 3 | Model Evaluation app | 45% | Required (Claude Code), see Section 3 |
| 4 | GitHub record of your work | 10% | Not needed |

- **Restricted resources**: <font color='red'>Sections 1 and 2: DO NOT use any generative AI tools. Finish them before you start Section 3.</font>
- **Recommended resources**: the Lab 08 notebook (holdout, cross validation, bootstrap, the confusion matrix and the cost matrix), the Lecture 06 slides on cost-benefit analysis, and your notes.

## Submission
- **Canvas**: upload this notebook, with your GitHub repository link filled in below.
- **GitHub**: push your work to your `comm4259-work` repository before the deadline (see Section 4).

**GitHub repository link:** *(paste it here, for example https://github.com/your-username/comm4259-work)*

**Due: 11:59 PM on Wednesday, November 4.**

## Before you start: put this assignment in your course repository

Do this first, before Section 1, so your work is in GitHub from the start and you do not have to remember it at the end.

1. Unzip `assignment_03.zip` into your `comm4259-work` folder, so this notebook and the data sit in `comm4259-work/assignment_03/`.
2. In VS Code, open the `comm4259-work` folder itself (File > Open Folder), then open this notebook from the Explorer panel.
3. In the VS Code terminal (Terminal > New Terminal), make your first commit now:
   ```
   git add .
   git commit -m "Assignment 3: start"
   git push
   ```
4. Repeat the three commands after each section. Section 4 explains what we look for in your repository.

# Section 1: Compare Validation Methods (25%, no generative AI)
In Lab 08 you estimated how well a decision tree does on new wines in three ways: holdout, cross validation and the bootstrap. In this section you use the same three methods on a new data set, the Titanic passengers, and see whether they agree on how deep the tree should be.

Use the prepared Titanic data, `titanic_train_clean.csv` (in this folder). It is prepared the same way for everyone: missing Age is filled with the mean, Embarked with the most frequent value and Fare with the median, and Sex, Embarked and Pclass are dummy-coded. The class label is `Survived` (1 = survived).

All the code is given. Run each cell in order and read the explanation above it. Step 1 shows training accuracy and Steps 2 to 4 show one method each, all for `max_depth = 10`; Step 5 runs all of them for the three depths you need for the table.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

In [2]:
titanic = pd.read_csv("titanic_train_clean.csv")
X = titanic.drop(["PassengerId", "Survived"], axis=1)   # PassengerId is only an ID
y = titanic["Survived"]
print(X.shape)
print(f"Baseline (everyone did not survive): {(y == 0).mean():.2%}")

(891, 9)
Baseline (everyone did not survive): 61.62%


The **baseline** is the accuracy of a "model" that always predicts the larger class (did not survive). A real model has to beat it to be useful.

## Step 1: The model, and why training accuracy is too optimistic
`make_tree(depth)` builds the same decision tree every time, so that only the **evaluation method** changes between steps. `criterion="entropy"` is the split rule from Lab 04, and `random_state=42` makes everyone's tree identical.

The cell then trains the tree on all 891 passengers and tests it on the **same** 891 passengers. The tree has already seen their answers, so this number is too optimistic: it measures memory, not prediction.

In [3]:
def make_tree(depth):
    return DecisionTreeClassifier(max_depth=depth, criterion="entropy", random_state=42)


def training_accuracy(depth):
    tree = make_tree(depth).fit(X, y)          # learn from all 891 passengers
    return accuracy_score(y, tree.predict(X))   # then test on the same 891


print(f"Training accuracy (max_depth=10): {training_accuracy(10):.2%}")

Training accuracy (max_depth=10): 91.36%


## Step 2: Holdout
Split the passengers once: 70% to train and 30% to test.
- `test_size=0.3` puts 30% of the passengers (268) in the test part.
- `stratify=y` keeps the same share of survivors in both parts.
- `random_state=1` fixes which passengers land in each part, so everyone gets the same split.

The tree learns from the 70% and is tested on the 30% it has never seen. The result depends on which passengers happen to land in the test part.

In [4]:
def holdout_accuracy(depth):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=1)
    tree = make_tree(depth).fit(X_train, y_train)        # learn from 70%
    return accuracy_score(y_test, tree.predict(X_test))  # test on the 30% it has not seen


print(f"Holdout accuracy (max_depth=10): {holdout_accuracy(10):.2%}")

Holdout accuracy (max_depth=10): 79.10%


## Step 3: 5-fold cross validation
Cut the passengers into 5 parts (folds). Each fold takes one turn as the test set while the other 4 folds train the tree, so **every** passenger is tested exactly once.
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=1)` makes the 5 folds, each with the same share of survivors.
- `cross_val_score` trains and tests 5 times and returns the 5 accuracies.
- We report their **mean** (the estimate) and their **standard deviation** (how much it moves from fold to fold).

In [5]:
def cv_accuracy(depth):
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    scores = cross_val_score(make_tree(depth), X, y, cv=folds)   # 5 accuracies, one per fold
    return scores.mean(), scores.std()


mean, sd = cv_accuracy(10)
print(f"Cross validation (max_depth=10): {mean:.2%} +/- {sd:.2%}")

Cross validation (max_depth=10): 80.25% +/- 2.17%


## Step 4: Bootstrap
Each round draws 891 passengers **with replacement** to train the tree: some passengers are picked twice or more, and about a third are not picked at all. The passengers who were not picked are the test set for that round (the "out-of-bag" passengers).
- `resample(rows, replace=True, n_samples=len(X), random_state=b)` draws the training rows for round `b`.
- `np.setdiff1d(rows, train_rows)` finds the rows that were not drawn.
- After 100 rounds we report the **mean** accuracy and the **middle 95%** of the 100 accuracies (from the 2.5th to the 97.5th percentile), which shows how much the estimate varies.

In [6]:
def bootstrap_accuracy(depth, rounds=100):
    rows = np.arange(len(X))
    scores = []
    for b in range(rounds):
        train_rows = resample(rows, replace=True, n_samples=len(X), random_state=b)  # draw with replacement
        test_rows = np.setdiff1d(rows, train_rows)                                   # passengers not drawn
        tree = make_tree(depth).fit(X.iloc[train_rows], y.iloc[train_rows])
        scores.append(accuracy_score(y.iloc[test_rows], tree.predict(X.iloc[test_rows])))
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


mean, low, high = bootstrap_accuracy(10)
print(f"Bootstrap (max_depth=10): {mean:.2%} (middle 95%: {low:.2%} to {high:.2%})")

Bootstrap (max_depth=10): 78.74% (middle 95%: 73.89% to 83.10%)


## Step 5: All three methods for three depths
This cell runs Steps 1 to 4 for `max_depth` 10, 8 and 3 and prints one row per depth. Copy these rows into the table in Exercise 1. The bootstrap takes a few seconds.

In [7]:
for depth in [10, 8, 3]:
    cv_mean, cv_sd = cv_accuracy(depth)
    b_mean, b_low, b_high = bootstrap_accuracy(depth)
    print(f"max_depth={depth:2d} | training {training_accuracy(depth):.2%} | holdout {holdout_accuracy(depth):.2%} | "
          f"CV {cv_mean:.2%} +/- {cv_sd:.2%} | bootstrap {b_mean:.2%} ({b_low:.2%} to {b_high:.2%})")

max_depth=10 | training 91.36% | holdout 79.10% | CV 80.25% +/- 2.17% | bootstrap 78.74% (73.89% to 83.10%)
max_depth= 8 | training 88.44% | holdout 77.99% | CV 81.14% +/- 3.19% | bootstrap 79.66% (74.72% to 83.31%)
max_depth= 3 | training 82.27% | holdout 80.22% | CV 81.14% +/- 2.79% | bootstrap 80.97% (75.56% to 84.93%)


## Exercise 1: Compare the methods (25 points)

<div style="background-color:#DFF5DC; color:#14532D; border:1px solid #A5D6A7; border-left:5px solid #2E7D32; padding:15px; margin-bottom:20px; border-radius:4px;">
<b>Exercise 1:</b> Copy the Step 5 results into the table below (15 points, 5 per depth). Then answer these three questions in one or two sentences each (10 points).
<ol>
<li>Which depth would each method pick: holdout, cross validation and bootstrap? Do they agree? If two depths tie, say which you would choose and why.</li>
<li>For one depth, the holdout accuracy is quite different from the cross validation accuracy. Why can a single holdout split mislead you?</li>
<li>Which method would you use to choose the depth for a real model, and why?</li>
</ol>
</div>

| max_depth | Training | Holdout | Cross validation (mean +/- sd) | Bootstrap (mean, middle 95%) |
|---|---|---|---|---|
| 10 | 91.36% | 79.10% | 80.25% +/- 2.17% | 78.74% (73.89% to 83.10%) |
| 8 | 88.44% | 77.99% | 81.14% +/- 3.19% | 79.66% (74.72% to 83.31%) |
| 3 | 82.27% | 80.22% | 81.14% +/- 2.79% | 80.97% (75.56% to 84.93%) |

Your answers:
1. A. Holdout would choose max_depth = 3 B. Cross validation would choose max_depth = 8 or = 3. Both are equal, but I would choose = 3 because it's less complex and more interperatable C. Bootstrap would choose max_depth = 3.
2. Because it's only testing one fold or one slice of the data out of the entire train-test set. The test set could have a disproportionately hard data point to predict compared to other random folds and that would make it penalize the holdout results on that fold. 
3. I would use cross validation because it makes sure to randomly partition and test 4 + 1 groups of the data so you get a very balanced result. This reduces variance and gives a very stable, reliable, and unbiased/less generalized result. 

# Section 2: Cost-Benefit Analysis (20%, no generative AI)
Accuracy treats every mistake the same. A business does not: missing a fraudulent account and auditing an honest one cost very different amounts. In this section you turn costs into a **cost matrix** and use it to choose among four fraud models, the case from Lecture 06.

## The case
Wells Fargo paid a **\$100 million** fine to the CFPB's Civil Penalty Fund over fraudulent accounts: 1.5 million deposit accounts and 565,000 credit card accounts. The bank wants to use predictive analytics to decide which accounts to audit.
- According to the CFPB, the median cost of a fraudulent account is **\$50,000**.
- According to Wells Fargo, auditing one account costs **\$1,000**. An audit of a fraudulent account catches the fraud, so the \$50,000 loss is avoided.
- **Current practice:** do nothing (no audits).
- **Analytics-driven practice:** audit every account the model predicts as **Fraud**; do nothing with accounts predicted as **Legit**.

## How to read a cost matrix relative to current practice
Each cell gives, for **one account** of that kind, how much more the analytics-driven practice costs than the current practice:

**cell = cost under the analytics-driven practice - cost under the current practice**

We write costs as positive numbers and savings as negative numbers. A negative cell is a benefit.

**The Predicted Legit column is 0, and it is given.** When the model says Legit, the bank does nothing, which is exactly what it does today. Nothing changes, so the difference is \$0 for both kinds of account:
- *Actual Fraud, Predicted Legit:* the analytics practice does nothing and loses \$50,000; the current practice also does nothing and loses \$50,000. The difference is \$50,000 - \$50,000 = **\$0**.
- *Actual Legit, Predicted Legit:* both practices do nothing and spend \$0. The difference is **\$0**.

**The Predicted Fraud column is yours.** When the model says Fraud, the bank audits the account, which it does not do today. For each of the two cells, ask two questions: what does this account cost the bank under the analytics practice (the audit, plus any fraud loss that is left), and what does it cost today (no audit, but any fraud goes undetected)? Then subtract.

| Cost per account (analytics minus current) | Predicted Fraud (audit) | Predicted Legit (do nothing) |
|---|---|---|
| **Actual Fraud** | ? | \$0 |
| **Actual Legit** | ? | \$0 |

## Exercise 2a: Complete the Predicted Fraud column

<div style="background-color:#DFF5DC; color:#14532D; border:1px solid #A5D6A7; border-left:5px solid #2E7D32; padding:15px; margin-bottom:20px; border-radius:4px;">
<b>Exercise 2a (8 points):</b> Fill in the two cells of the Predicted Fraud column below. For each cell, show the subtraction in one line (cost under the analytics practice minus cost today), as in the worked examples above.
</div>

| Cost per account (analytics minus current) | Predicted Fraud (audit) | Predicted Legit (do nothing) |
|---|---|---|
| **Actual Fraud** | -$49000 | \$0 |
| **Actual Legit** | $1000 | \$0 |

Your subtractions:
- Actual Fraud, Predicted Fraud:
- Actual Legit, Predicted Fraud:

## Exercise 2b: Plug your numbers into the code
The bank tested four models on 500 accounts: 190 fraudulent and 310 legitimate. Their confusion matrices are below (rows are the **actual** class, columns the **predicted** class, as in Lab 08).

| Model | Actual Fraud, Predicted Fraud | Actual Fraud, Predicted Legit | Actual Legit, Predicted Fraud | Actual Legit, Predicted Legit |
|---|---|---|---|---|
| M1 | 150 | 40 | 60 | 250 |
| M2 | 145 | 45 | 5 | 305 |
| M3 | 130 | 60 | 20 | 290 |
| M4 | 120 | 70 | 40 | 270 |

The code below computes four numbers for each model, with **Fraud as the positive class**:
- **Accuracy** = (correct predictions) / (all 500 accounts) = (TP + TN) / 500
- **Precision (Fraud)** = TP / (TP + FP): of the accounts we audit, the share that are really fraud
- **Recall (Fraud)** = TP / (TP + FN): of the fraudulent accounts, the share that we catch
- **Net cost** = the sum over the four cells of (number of accounts in the cell) x (cost per account in the cell). In code this is `(cm * cost).sum()`: multiply the two matrices cell by cell, then add the four products. A negative net cost is a net saving compared with doing nothing.

<div style="background-color:#DFF5DC; color:#14532D; border:1px solid #A5D6A7; border-left:5px solid #2E7D32; padding:15px; margin-bottom:20px; border-radius:4px;">
<b>Exercise 2b (4 points):</b> In the cell below, replace the two <code>0</code> values marked <code>&lt;-- your answer</code> with your Exercise 2a numbers (in dollars, no commas or \$ sign; a saving is negative). Do not change the Predicted Legit column. Run the cell; the table it prints is your answer.
</div>

In [8]:
# Cost per account, analytics-driven practice minus current practice (savings are negative)
#                 Predicted Fraud  Predicted Legit
cost = np.array([[-49000,               0],     # Actual Fraud   <-- your answer in the first column
                 [1000,               0]])    # Actual Legit   <-- your answer in the first column

# Confusion matrices: rows = actual (Fraud, Legit), columns = predicted (Fraud, Legit)
models = {
    "M1": np.array([[150, 40], [60, 250]]),
    "M2": np.array([[145, 45], [5, 305]]),
    "M3": np.array([[130, 60], [20, 290]]),
    "M4": np.array([[120, 70], [40, 270]]),
}

rows = []
for name, cm in models.items():
    TP, FN = cm[0, 0], cm[0, 1]   # actual Fraud: caught, missed
    FP, TN = cm[1, 0], cm[1, 1]   # actual Legit: audited for nothing, left alone
    rows.append({
        "Model": name,
        "Accuracy": f"{(TP + TN) / cm.sum():.2%}",
        "Precision (Fraud)": f"{TP / (TP + FP):.2%}",
        "Recall (Fraud)": f"{TP / (TP + FN):.2%}",
        "Net cost ($)": f"{(cm * cost).sum():,}",
    })
pd.DataFrame(rows)

,Model,Accuracy,Precision (Fraud),Recall (Fraud),Net cost ($)
0,M1,80.00%,71.43%,78.95%,"-7,290,000"
1,M2,90.00%,96.67%,76.32%,"-7,100,000"
2,M3,84.00%,86.67%,68.42%,"-6,350,000"
3,M4,78.00%,75.00%,63.16%,"-5,840,000"


## Exercise 2c: Recommend a model

<div style="background-color:#DFF5DC; color:#14532D; border:1px solid #A5D6A7; border-left:5px solid #2E7D32; padding:15px; margin-bottom:20px; border-radius:4px;">
<b>Exercise 2c (8 points):</b> Answer in the cell below.
<ol>
<li>Which model has the highest accuracy? Which has the lowest (most negative) net cost? (2 points)</li>
<li>Which model would you recommend for deployment, and why? Explain the difference between the two models in question 1 using their confusion matrices and your cost matrix: what does one model catch or avoid that the other does not, and what is that worth? (6 points)</li>
</ol>
</div>

Your answers:
1. Model M2 has the highest accuracy. Model M1 has the lowest/most negative cost. 
2. One could argue that you can deploy model M1 or M2. I might argue slightly in favor of M2. Recall is most important in fraud cases because it penalizes false negatives the most and if we let an account through that is actually fraudulent we will get fined $50k vs just paying $1k to audit the account. Therefore, we need to prioritize recall over precision and see that M1 has the highest Recall, but M2 is not too far behind and is cheaper and has higher accuracy (by 10 whole percent) so I will choose M2 in the end. 

# Section 3: Model Evaluation App (45%)
Build a small web app with one tab, **Model Evaluation**, so someone can see how the way a model is evaluated changes what they conclude about it. Build it the way you built the Assignment 2 workbench: **Claude Code** with your UVA account, started from `comm4259-work`, plan mode first, one session, and one `/export` at the end. Your `comm4259-work/CLAUDE.md` from Assignment 2 applies. You must be able to explain every line of code in your app.

```
comm4259-work/
└── assignment_03/
    ├── Assignment_03-Model_Evaluation.ipynb   this notebook
    ├── titanic_train_clean.csv
    └── dashboard/
        ├── app.py              the Streamlit app
        ├── requirements.txt    the packages the app needs
        ├── README.md           what the app shows and how to run it
        ├── AI_USE_LOG.md       how you used Claude Code (you write it)
        ├── ai_sessions/        your exported Claude Code session
        └── data/               a copy of titanic_train_clean.csv
```

<div style="background-color:#DFF5DC; color:#14532D; border:1px solid #A5D6A7; border-left:5px solid #2E7D32; padding:15px; margin-bottom:20px; border-radius:4px;">
<b>Build steps:</b>
<ol>
<li>In the VS Code terminal, from <code>comm4259-work</code>: <code>mkdir assignment_03/dashboard</code>, then <code>mkdir assignment_03/dashboard/data</code>, and copy the data into it: <code>cp assignment_03/titanic_train_clean.csv assignment_03/dashboard/data/</code>. Check your Streamlit version with <code>streamlit --version</code>.</li>
<li>Start <code>claude</code> from <code>comm4259-work</code>. <b>Keep this one session open until the app is finished</b>, and open a second terminal with the <b>+</b> whenever you need one. Press <b>Shift+Tab</b> until the bottom line shows <b>plan mode</b>, then paste the prompt below with your Streamlit version filled in.</li>
<li>Read the plan before you approve it, then let it build. Run the app in the second terminal: <code>streamlit run assignment_03/dashboard/app.py</code>. If it shows an error, paste the error into the Claude terminal and ask for a fix.</li>
<li><b>Check two numbers (no AI).</b> In the app, choose max_depth = 3 and 5-fold cross validation, then max_depth = 3 and a 30% holdout. Compare both with your Section 1 table, and write them in <code>AI_USE_LOG.md</code>: what the app shows, what your table shows, and whether they match. If a number does not match, find out why. Holdout and cross validation should match your table exactly; a bootstrap result can differ slightly if the app uses other random seeds.</li>
<li>Export the session once, at the end, in the Claude terminal: <code>/export assignment_03/dashboard/ai_sessions/assignment_03.md</code>. Then type <code>/exit</code>.</li>
</ol>
</div>

Prompt for step 2:

```
Work only in assignment_03/dashboard/: create and edit files only there.
My Streamlit version is <your version>. Only use features available in that version, and do not install or upgrade any packages.
Plan a Streamlit app: app.py (read the data with Path(__file__).parent / "data") with one tab, Model Evaluation,
plus requirements.txt (include pyarrow<17), README.md, an empty ai_sessions/ folder and an empty AI_USE_LOG.md (I write the log myself).
Use data/titanic_train_clean.csv. The target is Survived and the positive class is 1. Drop PassengerId.
The tab must let a user:
1. set max_depth with a slider (1 to 15) for DecisionTreeClassifier(criterion="entropy", random_state=42);
2. choose a validation method and see the accuracy: holdout (test-size slider from 0.10 to 0.50, default 0.30,
   passed to train_test_split as a fraction, stratified, random_state=1),
   k-fold cross validation (k slider 3 to 10, StratifiedKFold with shuffle=True and random_state=1, mean and
   standard deviation), or bootstrap (rounds slider 10 to 200, round b uses resample(..., random_state=b),
   out-of-bag accuracy, mean and the middle 95%);
3. see the confusion matrix on the same holdout split as item 2 (rows actual, columns predicted, Survived = 1 first),
   with precision, recall and F1 for the positive class;
4. enter the cost of a false positive and of a false negative, and see the total cost of the holdout predictions.
Show every rate as a percentage with two decimals, and show the scikit-learn line each result comes from.
The app makes no decisions on its own and uses no AI model. Keep the code simple enough for a beginner to read.
Ask me if anything in this plan is unclear.
```

| Item | Points |
|---|---|
| The app runs from a fresh copy of your repository with `streamlit run assignment_03/dashboard/app.py` | 15 |
| The tab does the four things in the prompt (max_depth, the three validation methods, confusion matrix with precision, recall and F1, total cost), and they work correctly | 15 |
| `README.md` explains what the tab shows, how to run the app, and why the holdout and cross validation accuracies can differ | 8 |
| `AI_USE_LOG.md` has the assistant, your main prompts, what it got wrong or what you changed, the time you spent and the two checked numbers; the exported session is in `assignment_03/dashboard/ai_sessions/assignment_03.md` | 7 |

# Section 4: GitHub Record of Your Work (10%)

Keep this assignment in your `comm4259-work` repository, the same one you push labs to.

1. **Folder.** Unzip the assignment into `comm4259-work/assignment_03/`. The app goes in `comm4259-work/assignment_03/dashboard/`.
2. **Commit as you go.** Make **at least three commits** for this assignment, spread over your work (for example after each section), each with a message that says what changed. Run the commands from `comm4259-work`:
   ```
   git add .
   git commit -m "Assignment 3: finished Section 1"
   git push
   ```
3. **Push before the deadline.** Your last commit must be pushed before the Canvas due date. Commit times are checked.
4. **Teaching team access.** Keep `analyticsli`, `zayliet` and `jisungl` as collaborators on your repository (you added them in Assignment 1).
5. **Paste your repository link** in the Description section at the top of this notebook.

| Item | Points |
|---|---|
| At least three commits for this assignment, spread over the work, pushed before the deadline | 5 |
| Folders as described above, your repository link at the top of this notebook, and the teaching team able to see the repository | 5 |